In [1]:
# =====================================================================
# Part II - Image Colorization - TEMPLATE
# =====================================================================
#
# Task: take a grayscale image and predict its colors.
#
# You build and train the model any way you want (autoencoder, VAE, GAN, ...).
#
# The grader will:
#   1. run your model class + load_model() to load your saved weights,
#   2. call colorize() on their own images,
#   3. compare your output to hidden color images (PSNR / MSE).
#
# So you MUST keep the 3 fixed rules below.
# =====================================================================
#
# ---------------------------------------------------------------------
# FIXED RULES (do not change)
# ---------------------------------------------------------------------
#
# 1. Save your trained weights as a state_dict:
#        torch.save(model.state_dict(), "weights.pth")
#    Submit this "weights.pth" file together with your notebook.
#
# 2. All images are 256 x 256 PNG.
#    colorize() input  : grayscale array, shape (256, 256), float in [0, 1]
#    colorize() output : RGB array,       shape (256, 256, 3), float in [0, 1]
#
# 3. Do file loading OUTSIDE colorize() (see the demo at the bottom).
#    colorize() only takes arrays, not file paths.
# ---------------------------------------------------------------------

In [2]:
import numpy as np
import torch
import torch.nn as nn

IMG_SIZE = 256

In [ ]:
# ---------------------------------------------------------------------
# 1) YOUR MODEL
# ---------------------------------------------------------------------
# A U-Net that predicts only CHROMINANCE (Cb, Cr), not full RGB.
#
# Plain RGB regression with L1/MSE loss is known to produce muted,
# undersaturated colors: the model hedges toward "safe" averaged
# colors whenever it's unsure, because errors in R/G/B are entangled
# with brightness errors too. The standard fix in the colorization
# literature is to work in a luma/chroma colorspace (Y/Cb/Cr): the
# grayscale input IS (almost exactly) the Y channel already, so it
# doesn't need to be predicted at all -- only the 2 color channels
# (Cb, Cr) do. This confines all prediction error to color, never
# brightness, which is both an easier learning problem and closer to
# how the eye actually perceives color images.
#
# base=64 with residual blocks (measured: 15.3M params, ~3.5x the
# previous base=24 plain U-Net's 4.4M): sized and shaped to match a
# second, independently-trained reference implementation on a more
# diverse dataset (Kaggle "Natural Images": airplane/car/cat/dog/
# flower/fruit/motorbike/person) whose weights are demonstrably
# capable of vivid, non-muted colorization on real photos, verified
# directly here by loading its state_dict and running it on our own
# local test images before committing to copying its design. Matching
# its architecture exactly (see ResBlock below: conv-bn-relu-conv-bn +
# a 1x1-conv-bn shortcut when channel counts differ, added back before
# the final ReLU) means those weights can be loaded as a warm start
# for the encoder/decoder body instead of training from scratch on
# this bigger network from random init -- real GPU time already spent
# is not wasted. Only the final 1x1 output layer is excluded from the
# warm start (see the training cell below): its reference used a
# different output convention (tanh-bounded U/V in YUV space, computed
# inside forward()) than ours (sigmoid-bounded Cb/Cr, kept below
# unchanged -- it is a proven, working interface, no reason to touch
# it or the colorize()/load_model() cells that depend on it).
#
# The default here MUST match what's actually trained -- load_model()
# below rebuilds with no arguments, so a mismatched default would fail
# to load the saved weights.

Y_R, Y_G, Y_B = 0.299, 0.587, 0.114  # matches PIL's L = ITU-R 601-2 luma


def rgb_to_ycbcr(rgb):
    """rgb: (..., H, W, 3) in [0,1] -> y, cb, cr each (..., H, W) in [0,1]."""
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    y = Y_R * r + Y_G * g + Y_B * b
    cb = -0.168736 * r - 0.331264 * g + 0.5 * b + 0.5
    cr = 0.5 * r - 0.418688 * g - 0.081312 * b + 0.5
    return y, cb, cr


def ycbcr_to_rgb(y, cb, cr):
    """y, cb, cr: (..., H, W) in [0,1] -> rgb (..., H, W, 3) in [0,1]."""
    cb0, cr0 = cb - 0.5, cr - 0.5
    r = y + 1.402 * cr0
    g = y - 0.344136 * cb0 - 0.714136 * cr0
    b = y + 1.772 * cb0
    return np.clip(np.stack([r, g, b], axis=-1), 0.0, 1.0)


class ResBlock(nn.Module):
    """conv-bn-relu-conv-bn, plus a projection shortcut when the
    channel count changes, added back before the final ReLU. Lets
    gradients skip the two convs directly (classic ResNet benefit:
    trains more reliably at this depth than plain stacked convs), and
    is the exact block shape needed to warm-start from the reference
    weights described above."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Sequential()

    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + self.shortcut(x))


class ColorizeModel(nn.Module):
    def __init__(self, base=64):
        super().__init__()
        self.enc1 = ResBlock(1, base)
        self.enc2 = ResBlock(base, base * 2)
        self.enc3 = ResBlock(base * 2, base * 4)
        self.enc4 = ResBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ResBlock(base * 8, base * 8)

        self.upconv4 = nn.ConvTranspose2d(base * 8, base * 8, kernel_size=2, stride=2)
        self.dec4 = ResBlock(base * 16, base * 4)
        self.upconv3 = nn.ConvTranspose2d(base * 4, base * 4, kernel_size=2, stride=2)
        self.dec3 = ResBlock(base * 8, base * 2)
        self.upconv2 = nn.ConvTranspose2d(base * 2, base * 2, kernel_size=2, stride=2)
        self.dec2 = ResBlock(base * 4, base)
        self.upconv1 = nn.ConvTranspose2d(base, base, kernel_size=2, stride=2)
        self.dec1 = ResBlock(base * 2, base)

        self.final_conv = nn.Conv2d(base, 2, 1)  # Cb, Cr only

    def forward(self, x):
        # x: (batch, 1, 256, 256) -- the Y (luminance) channel
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.upconv4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.upconv3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.upconv1(d2), e1], dim=1))
        return torch.sigmoid(self.final_conv(d1))  # (batch, 2, 256, 256): cb, cr in [0,1]

In [4]:
# ---------------------------------------------------------------------
# 2) LOAD YOUR TRAINED WEIGHTS
# ---------------------------------------------------------------------
# Rebuilds the empty model and loads the saved numbers.
# This is instant - no training.
def load_model(weights_path="weights.pth"):
    model = ColorizeModel()
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model

In [ ]:
# ---------------------------------------------------------------------
# 3) COLORIZE ONE IMAGE
# ---------------------------------------------------------------------
# gray_img: numpy array, shape (256, 256), float in [0, 1]
# returns : numpy array, shape (256, 256, 3), float in [0, 1]
#
# The external contract is unchanged (grayscale in, RGB out) -- the
# Y/Cb/Cr split is purely an internal implementation detail. gray_img
# is used directly as Y (it already IS the luminance channel), the
# model predicts Cb/Cr, and the three are recombined into RGB.
def colorize(gray_img, model):
    x = torch.from_numpy(gray_img).float().view(1, 1, IMG_SIZE, IMG_SIZE)

    with torch.no_grad():           # no gradients needed for inference
        cb_cr = model(x)            # (1, 2, 256, 256)

    cb = cb_cr[0, 0].cpu().numpy()
    cr = cb_cr[0, 1].cpu().numpy()
    return ycbcr_to_rgb(gray_img, cb, cr)

---
### Everything below this point is the training/reproducibility record

Dataset download and the real training loop (COCO + Natural Images +
Flowers102, ~100k images, patience-based early stopping) -- this is
*how* `weights.pth` was actually produced, mirroring
`dev/train_local_gpu.py` (the script that really ran, on a local GPU),
kept for the write-up and the viva. Not required for grading, and
automatically skipped entirely below whenever `weights.pth` is already
present (this repo's actual submitted state) -- nothing here downloads
or retrains just from running the notebook.

In [ ]:
# =======================================================================
# TRAINING -- produces weights.pth
# =======================================================================
# Everything above this line is the fixed submission interface. Below is
# how weights.pth was ACTUALLY produced: this mirrors dev/train_local_gpu.py
# exactly (dataset sources, hue balancing, scheduler, early stopping) --
# that script is what really ran, on a local GPU (Colab's session limits
# don't fit a run this size, and COCO alone is a ~19GB one-time download).
# This cell documents that real process rather than a smaller, different
# pipeline, but is skipped entirely below whenever weights.pth already
# exists (this repo's actual submitted state) -- nothing needs to
# download or retrain to grade an already-trained model.
#
# No training images were provided for this assignment ("you may choose
# all the images you want to train your model" -- directives.txt).
#
# THREE data sources:
#   1. Kaggle "Natural Images" (~6,900 photos, 8 categories) -- the base.
#   2. Flowers102 in full (8,189 images) -- color diversity.
#   3. Up to COCO_IMAGES photos from COCO train2017 (118,287 total) --
#      added for scene diversity (sky, water, grass, streets, people)
#      the flower-heavy pool was thin on, especially blue and red
#      (measured: ~2.8% of the flower pool is dominant-blue vs. COCO
#      photos routinely being 20-30% sky/water). This is what actually
#      grew training to the ~100k-image run that produced the submitted
#      weights.pth -- far more than an earlier, smaller version of this
#      cell used.
#
# Every image across all three sources is scored for its dominant hue,
# and training draws from all of them with weights inversely
# proportional to how common that hue already is in the pool -- inverse-
# frequency class-imbalance handling applied to color instead of a
# label, so no single common color dominates the gradient just because
# it's the most frequent one in the raw data.

import os
import sys
import glob
import random
import zipfile
import subprocess
import sysconfig
import shutil
import torch.utils.data
import torchvision
import torchvision.transforms as T
from PIL import Image
from collections import Counter

COCO_IMAGES = 80000  # matches dev/train_local_gpu.py's --coco-images default, the value the real run used
DATA_ROOT = "colorization_data"
os.makedirs(DATA_ROOT, exist_ok=True)

if os.path.exists("weights.pth"):
    print("weights.pth already present -- skipping all data download and hue "
          "classification below entirely (train_ds/val_ds are NOT created in "
          "this run). Delete weights.pth "
          "first if you actually want to reproduce training from scratch.")
else:
    NATURAL_IMAGES_DIR = os.path.join(DATA_ROOT, "natural-images")
    if not os.path.isdir(NATURAL_IMAGES_DIR):
        try:
            from google.colab import drive as _drive  # noqa: F401  (proxy for "are we in Colab")
            IN_COLAB = True
        except ImportError:
            IN_COLAB = False

        if not IN_COLAB:
            raise RuntimeError(
                "Not running in Colab and no local copy of Natural Images found at %s. "
                "Run this in Colab so the Kaggle download below can run, or place the "
                "extracted dataset at that path yourself." % NATURAL_IMAGES_DIR)

        kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
        if not os.path.exists(kaggle_json):
            print("No Kaggle credentials found at ~/.kaggle/kaggle.json.")
            print("Upload your kaggle.json now (kaggle.com -> account -> Create New API Token):")
            from google.colab import files
            uploaded = files.upload()
            os.makedirs(os.path.dirname(kaggle_json), exist_ok=True)
            for fname in uploaded:
                if fname.endswith(".json"):
                    shutil.move(fname, kaggle_json)
            os.chmod(kaggle_json, 0o600)
        # sys.executable / sysconfig, not bare "pip"/"kaggle": a bare command
        # can fail to resolve right after installing (confirmed directly
        # during local GPU setup this project already went through).
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        kaggle_exe = os.path.join(sysconfig.get_path("scripts"), "kaggle")
        subprocess.run([kaggle_exe, "datasets", "download", "-d", "prasunroy/natural-images",
                         "-p", DATA_ROOT], check=True)
        zip_path = os.path.join(DATA_ROOT, "natural-images.zip")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(NATURAL_IMAGES_DIR)
        print("Natural Images dataset downloaded.")

    natural_paths = sorted(glob.glob(os.path.join(NATURAL_IMAGES_DIR, "**", "*.jpg"), recursive=True))
    if not natural_paths:
        raise RuntimeError(
            "Natural Images directory exists at %s but no .jpg files were found in it -- "
            "the download/unzip likely produced a different folder layout than expected. "
            "Check the actual contents of that directory." % NATURAL_IMAGES_DIR)
    print("natural-images found:", len(natural_paths))

    flowers_dir = os.path.join(DATA_ROOT, "flowers-102", "jpg")
    if not os.path.isdir(flowers_dir):
        for split in ["train", "val", "test"]:
            torchvision.datasets.Flowers102(root=DATA_ROOT, split=split, download=True)
    flower_paths = sorted(glob.glob(os.path.join(flowers_dir, "*.jpg")))
    print("flowers102 found:", len(flower_paths))

    # COCO train2017 -- see the module docstring above for why. Ships as one
    # ~19GB zip regardless of COCO_IMAGES (no partial-file download for a
    # subset); only COCO_IMAGES members get extracted, chosen by a fixed-
    # seed random sample so re-runs extract the same files instead of a new
    # random subset each time. COCO_IMAGES=0 skips this source entirely.
    COCO_TRAIN2017_URL = "http://images.cocodataset.org/zips/train2017.zip"
    coco_dir = os.path.join(DATA_ROOT, "coco-train2017")
    os.makedirs(coco_dir, exist_ok=True)
    coco_paths = sorted(glob.glob(os.path.join(coco_dir, "*.jpg")))
    if COCO_IMAGES > 0 and len(coco_paths) < COCO_IMAGES:
        coco_zip = os.path.join(DATA_ROOT, "coco_train2017.zip")
        if not os.path.exists(coco_zip):
            print("downloading COCO train2017 (~19GB, one-time) ...")
            subprocess.run(["curl", "-L", "-o", coco_zip, COCO_TRAIN2017_URL], check=True)
        print("extracting up to %d COCO images ..." % COCO_IMAGES)
        with zipfile.ZipFile(coco_zip) as zf:
            members = [m for m in zf.namelist() if m.endswith(".jpg")]
            random.Random(0).shuffle(members)
            for m in members[:COCO_IMAGES]:
                target = os.path.join(coco_dir, os.path.basename(m))
                if not os.path.exists(target):
                    with zf.open(m) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)
        coco_paths = sorted(glob.glob(os.path.join(coco_dir, "*.jpg")))
    print("coco found:", len(coco_paths))

    # --- Dominant-hue classification, every image, all three sources ---
    # 8 equal 45-degree hue bins around the color wheel, plus "neutral" for
    # images that are mostly desaturated (most airplane/car/person photos,
    # and most COCO street/sky scenes too: metal, sky, pavement, skin --
    # no single hue dominates).
    HUE_BIN_EDGES = [(345, 15, "red"), (15, 45, "orange"), (45, 75, "yellow"),
                      (75, 165, "green"), (165, 195, "cyan"), (195, 255, "blue"),
                      (255, 285, "purple"), (285, 345, "magenta")]
    NEUTRAL_THRESHOLD = 0.12  # mean saturation*value below this -> "neutral"

    def dominant_hue_bin(path):
        """Which of the 8 hue bins (or 'neutral') this image is mostly
        made of, weighted by saturation*value so washed-out pixels barely
        count -- this is about the color that's actually vivid in the
        photo, not a technically-present trace of it."""
        img = Image.open(path).convert("RGB").resize((64, 64))
        arr = np.asarray(img).astype(np.float32) / 255.0
        r, g, b = arr[..., 0], arr[..., 1], arr[..., 2]
        maxc, minc = np.max(arr, axis=-1), np.min(arr, axis=-1)
        v = maxc
        s = np.where(maxc > 0, (maxc - minc) / np.where(maxc == 0, 1, maxc), 0)
        delta = maxc - minc + 1e-8
        hue = np.zeros_like(maxc)
        mask_r = (maxc == r)
        mask_g = (maxc == g) & ~mask_r
        mask_b = (maxc == b) & ~mask_r & ~mask_g
        hue[mask_r] = (60 * (((g - b) / delta) % 6))[mask_r]
        hue[mask_g] = (60 * (((b - r) / delta) + 2))[mask_g]
        hue[mask_b] = (60 * (((r - g) / delta) + 4))[mask_b]
        weight = s * v
        if weight.mean() < NEUTRAL_THRESHOLD:
            return "neutral"
        best_bin, best_score = "neutral", 0.0
        for lo, hi, name in HUE_BIN_EDGES:
            band = (hue >= lo) | (hue <= hi) if lo > hi else (hue >= lo) & (hue <= hi)
            score = (weight * band).sum()
            if score > best_score:
                best_bin, best_score = name, score
        return best_bin

    hue_cache_path = os.path.join(DATA_ROOT, "hue_cache.json")

    def get_hue_bins_cached(paths, cache_path, save_every=2000):
        """Same result as [dominant_hue_bin(p) for p in paths], but reads a
        JSON cache (path -> bin) first and only classifies images not
        already in it -- with ~100k images across all three sources, this
        one-time scan is real minutes, and saving the cache periodically
        (not only at the end) means an interrupted run only re-classifies
        what's changed since the last save, not everything from zero."""
        cache = {}
        if os.path.exists(cache_path):
            with open(cache_path, encoding="utf-8") as f:
                cache = json.load(f)
        bins, new_count = [], 0
        for p in paths:
            if p in cache:
                bins.append(cache[p])
            else:
                b = dominant_hue_bin(p)
                cache[p] = b
                bins.append(b)
                new_count += 1
                if new_count % save_every == 0:
                    with open(cache_path, "w", encoding="utf-8") as f:
                        json.dump(cache, f)
        if new_count:
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(cache, f)
        print("hue cache: classified %d new image(s), reused %d from cache"
              % (new_count, len(paths) - new_count))
        return bins

    import json
    print("classifying dominant hue for every image (one-time cost, cached after the first run)...")
    all_paths = natural_paths + flower_paths + coco_paths
    all_bins = get_hue_bins_cached(all_paths, hue_cache_path)
    bin_counts = Counter(all_bins)
    print("hue distribution across the full pool:", dict(bin_counts))

    combined = list(zip(all_paths, all_bins))
    random.Random(0).shuffle(combined)
    image_paths = [p for p, _ in combined]
    image_bins = [b for _, b in combined]

    n_val = max(1, int(0.1 * len(image_paths)))
    val_paths, train_paths = image_paths[:n_val], image_paths[n_val:]
    val_bins, train_bins = image_bins[:n_val], image_bins[n_val:]
    print("combined dataset:", len(image_paths), " train:", len(train_paths), " val:", len(val_paths))

    # Inverse-SQUARE-ROOT-frequency sample weights for the TRAIN split only
    # -- softened (sqrt, not the full 1/count) after direct testing showed
    # pure inverse-frequency over-corrects: it pushed rare hues (e.g.
    # purple, backed by limited real diversity in the flower pool) hard
    # enough that the model started predicting that hue in unrelated,
    # ambiguous regions of genuinely novel test photos. Sqrt weighting
    # still corrects the imbalance, just less aggressively.
    train_bin_counts = Counter(train_bins)
    train_weights = [1.0 / train_bin_counts[b] ** 0.5 for b in train_bins]
    print("train hue distribution:", dict(train_bin_counts))

    class ColorizationDataset(torch.utils.data.Dataset):
        """Loads a color photo, returns (Y, [Cb, Cr]), all 256x256 in [0, 1].

        Y is the grayscale input (== luminance); Cb/Cr are the color
        channels the model must predict. The photo supervises itself --
        no separate labels needed. `augment=True` applies, train split
        only: a random horizontal flip, a random-resized-crop, and mild
        brightness/contrast jitter on Y -- for free extra variety from the
        same images.

        Flip and crop are applied to the COLOR image first, and Y/Cb/Cr
        are derived from that SAME already-transformed image -- not
        touching one side only, since Y and Cb/Cr are mathematically
        coupled (both come from the same original RGB pixel).

        Brightness/contrast jitter is the deliberate exception: applied to
        Y alone, Cb/Cr left untouched -- a reasonable approximation of a
        real exposure change (chroma staying roughly stable across
        lighting is YCbCr's whole design point), kept mild (+-15%) and
        clamped to [0, 1].

        Hue/saturation jitter (changing the color itself) was deliberately
        NOT added: it would teach the model that a hue-shifted color is
        sometimes "correct," undermining the one thing colorization is
        supposed to learn.
        """

        def __init__(self, paths, size=IMG_SIZE, augment=False):
            self.paths = paths
            self.size = size
            self.resize = T.Resize((size, size))
            self.random_resized_crop = T.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.9, 1.1))
            self.augment = augment

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, idx):
            img = Image.open(self.paths[idx]).convert("RGB")
            if self.augment:
                img = self.random_resized_crop(img)
                if random.random() < 0.5:
                    img = img.transpose(Image.FLIP_LEFT_RIGHT)
            else:
                img = self.resize(img)
            rgb = np.array(img).astype("float32") / 255.0
            y, cb, cr = rgb_to_ycbcr(rgb)
            if self.augment:
                brightness = random.uniform(0.85, 1.15)
                contrast = random.uniform(0.85, 1.15)
                y = np.clip((y - 0.5) * contrast + 0.5, 0.0, 1.0)
                y = np.clip(y * brightness, 0.0, 1.0)
            y_t = torch.from_numpy(y).float().unsqueeze(0)
            cbcr_t = torch.from_numpy(np.stack([cb, cr], axis=0)).float()
            return y_t, cbcr_t

    train_ds = ColorizationDataset(train_paths, augment=True)
    val_ds = ColorizationDataset(val_paths, augment=False)

In [ ]:
import torchvision

# True whenever a trained weights.pth already sits next to this notebook
# -- the actual state this repo is submitted in. Checked up front, before
# Drive or anything else below is touched (see the dataset cell above too
# -- it checks the same thing independently, so nothing downloads either).
SKIP_TRAINING = os.path.exists("weights.pth")

BATCH_SIZE = 8   # matches dev/train_local_gpu.py -- proven not to OOM on
                 # an RTX 4050 (6GB VRAM) with the VGG perceptual loss active.
LR = 1e-3
PATIENCE = 8     # stop once val loss hasn't improved for this many epochs --
                 # the real run's actual stopping condition (this notebook
                 # used to show a fixed 30-epoch cosine schedule instead,
                 # which is NOT what the real training used -- corrected
                 # during a submission review). Needs to be meaningfully
                 # larger than LR_PATIENCE below, so a just-halved LR gets a
                 # real chance to help before the run gives up.
LR_PATIENCE = 2  # halve the LR after this many epochs with no improvement
EPOCHS = 300     # a high ceiling PATIENCE is expected to trigger well before,
                 # not a target -- unlike the old fixed-epoch version, this
                 # doesn't cap the run at an arbitrary point with LR still high.

if SKIP_TRAINING:
    CHECKPOINT_DIR = "."
    print("weights.pth already present -- skipping Google Drive, the "
          "training loop, and (see the dataset cell above) the data "
          "download entirely.")
else:
    # Persist checkpoints to Google Drive, not Colab's local disk -- a
    # disconnect only costs the current, incomplete epoch, never anything
    # before it. "_v3" directory, not "_v2": this run's dataset changed
    # again (COCO added on top of Natural Images + Flowers102), so a v2
    # checkpoint reflects training on a different, much smaller pool --
    # kept separate so an old v2 checkpoint is never silently resumed
    # into this larger run.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_DIR = "/content/drive/MyDrive/colorization_checkpoints_v3"
    except ImportError:
        CHECKPOINT_DIR = "."  # not running in Colab (e.g. local test) -- use the working directory
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint.pt")
WEIGHTS_PATH = os.path.join(CHECKPOINT_DIR, "weights.pth")
WEIGHTS_BEST_PATH = os.path.join(CHECKPOINT_DIR, "weights_best.pth")
print("checkpoint directory:", CHECKPOINT_DIR)

torch.manual_seed(0)
model = ColorizeModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    # Every image is resized to a fixed 256x256 -- input size never varies,
    # which is exactly the condition this needs to be a safe, free speedup
    # (cuDNN benchmarks conv algorithms once for that fixed shape and
    # reuses the fastest one).
    torch.backends.cudnn.benchmark = True
model.to(device)

# WARM START (only on a completely fresh run -- once a v3 checkpoint.pt
# exists, RESUME below loads that instead; if SKIP_TRAINING already loaded
# weights.pth above, this whole block is skipped too).
#
# Loads THIS SAME architecture's own earlier checkpoint (epoch 26 of the
# pre-COCO run, dev/weights_epoch26_warm_start.pth) -- weights only,
# strict=True: same model, same output convention, nothing needs
# excluding this time (unlike an earlier version of this notebook, which
# warm-started from a *different*, independently-trained reference model
# with a different YUV/tanh output convention and had to exclude
# final_conv for that reason -- not the case here). A FRESH optimizer and
# LR schedule on top, not the old run's momentum: this run trains on a
# much larger and differently-composed dataset (+ up to COCO_IMAGES COCO
# photos, added for scene diversity the flower-heavy pool was thin on,
# especially blue), so the old optimizer state doesn't describe this
# run's actual data even though the learned weights are still worth
# keeping as a starting point.
WARM_START_WEIGHTS_PATH = os.path.join("..", "dev", "weights_epoch26_warm_start.pth")
if not SKIP_TRAINING and not os.path.exists(CHECKPOINT_PATH):
    if os.path.exists(WARM_START_WEIGHTS_PATH):
        sd = torch.load(WARM_START_WEIGHTS_PATH, map_location="cpu")
        model.load_state_dict(sd, strict=True)
        print("warm-started all weights from", WARM_START_WEIGHTS_PATH,
              "-- fresh optimizer/scheduler, epoch counter reset to 0")
    else:
        print("no warm-start weights found at %s -- starting from random "
              "initialization instead. Training will just need more epochs "
              "to reach the same quality." % WARM_START_WEIGHTS_PATH)
elif SKIP_TRAINING:
    print("SKIP_TRAINING -- warm start skipped, weights.pth will be loaded below instead")
else:
    print("existing v3 checkpoint found -- skipping warm start, "
          "RESUME below will load it instead")

opt = torch.optim.Adam(model.parameters(), lr=LR)
# ReduceLROnPlateau, not a fixed cosine schedule: PATIENCE (early stopping)
# is what actually ends the run, not a target epoch count, so a schedule
# tied to val-loss plateaus fits better than one tied to a step count that
# EPOCHS=300 is mostly just a high ceiling for, not a value meant to be
# reached.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=LR_PATIENCE)
# Mixed precision: on RTX-class GPUs (Tensor Cores), running the forward
# pass in float16 where safe is both faster and roughly halves activation
# memory, while GradScaler keeps the backward pass numerically stable (it
# scales the loss up before backward so small gradients don't underflow to
# zero in float16, then unscales before the optimizer step). enabled=False
# on CPU makes autocast/scaler a no-op automatically.
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
print("device:", device)


def ycbcr_to_rgb_torch(y, cb, cr):
    """Differentiable version of ycbcr_to_rgb (batched tensors, gradients
    flow through) -- needed to feed predicted colors into the perceptual
    loss below. Same math as the numpy version above, kept in sync."""
    cb0, cr0 = cb - 0.5, cr - 0.5
    r = y + 1.402 * cr0
    g = y - 0.344136 * cb0 - 0.714136 * cr0
    b = y + 1.772 * cb0
    return torch.clamp(torch.cat([r, g, b], dim=1), 0.0, 1.0)


def color_weighted_l1(pred_cbcr, target_cbcr, target_rgb):
    """Per-pixel L1 in Cb/Cr space, weighted up to 6x on pixels that are
    genuinely colorful in the ground truth and down-weighted toward 1x
    on near-gray pixels. Plain uniform L1 spends equal effort getting a
    gray sidewalk exactly right as it does getting a red flower right,
    which is exactly backwards for a metric that cares about color:
    with uniform weighting, a model can lower its average loss a lot by
    hedging every uncertain pixel toward a safe, muted average color,
    since most pixels in most photos are near-neutral anyway. Weighting
    by how colorful the pixel actually is forces the gradient to keep
    pushing on the pixels where getting the color right is the whole
    point, which is where the earlier "muted, undersaturated" failure
    mode came from.
    """
    gray_target = target_rgb.mean(dim=1, keepdim=True)
    colorfulness = torch.abs(target_rgb - gray_target).mean(dim=1, keepdim=True)
    weight = 1.0 + 5.0 * colorfulness
    return (torch.abs(pred_cbcr - target_cbcr) * weight).mean()


class VGGPerceptualLoss(nn.Module):
    """L1 distance between VGG19 features of predicted vs. true RGB,
    instead of raw pixels. Used ALONGSIDE the color-weighted L1 above,
    not instead of it: the grader scores PSNR/MSE directly, a pixel-
    level metric, so pixel accuracy still has to matter too. This adds
    a complementary signal about texture/structure that pixel losses
    alone don't capture.

    layer_idx=26 -- true relu4_4: VGG19's features Sequential is 5 conv
    blocks of [conv,relu]*2-or-4 + pool; block4 (512-channel convs) ends
    at index 26 = the 4th ReLU in that block.
    """
    def __init__(self, layer_idx=26):
        super().__init__()
        weights = torchvision.models.VGG19_Weights.IMAGENET1K_V1
        vgg = torchvision.models.vgg19(weights=weights).features[:layer_idx].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred_rgb, target_rgb):
        pred_n = (pred_rgb - self.mean) / self.std
        target_n = (target_rgb - self.mean) / self.std
        return nn.functional.l1_loss(self.vgg(pred_n), self.vgg(target_n))


# Only actually instantiated (loads VGG19's ImageNet weights) when there's
# training to do -- SKIP_TRAINING means nothing below uses it, so this
# skips a pointless download/load of VGG19's weights too.
vgg_loss_fn = VGGPerceptualLoss().to(device) if not SKIP_TRAINING else None
PERCEPTUAL_WEIGHT = 0.05  # color-weighted L1 dominates; perceptual nudges toward natural texture

# WeightedRandomSampler, not shuffle=True: train_weights (dataset cell) is
# inverse-frequency by dominant hue, so batches draw roughly equal expected
# exposure per hue per epoch regardless of how common that hue actually is
# in the combined pool. val_loader stays plain/unweighted on purpose -- it
# needs to reflect genuine expected performance on the real distribution.
# Skipped entirely when SKIP_TRAINING, since train_ds/val_ds don't even
# exist in that case (the dataset cell above skips creating them too).
if not SKIP_TRAINING:
    train_sampler = torch.utils.data.WeightedRandomSampler(
        train_weights, num_samples=len(train_weights), replacement=True)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
        num_workers=4, pin_memory=(device.type == "cuda"), persistent_workers=True)
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=(device.type == "cuda"), persistent_workers=True)

# RESUME: if a checkpoint from an earlier (possibly interrupted) run of
# this same cell exists -- on Drive, so it survives a full runtime reset,
# not just this session -- pick up from exactly where it left off: model
# weights, optimizer momentum, scheduler and scaler state, instead of
# silently starting over. Takes priority over the warm start above.
#
# SKIP_TRAINING takes priority over both: load weights.pth directly and
# skip straight to "ready for the cells below" without touching
# checkpoint.pt at all. (A fresh clone of this repo -- a grader's machine,
# or this notebook rerun from scratch -- has weights.pth but never
# checkpoint.pt, deliberately not committed at 180MB+; an earlier version
# of this logic didn't account for that and would have either crashed or
# silently launched a brand-new run instead of using the already-trained
# weights sitting right next to it -- a real correctness bug, not a
# training-quality concern, caught during a submission review.)
start_epoch = 0
best_val_loss = float("inf")
if SKIP_TRAINING:
    model.load_state_dict(torch.load("weights.pth", map_location=device))
    print("loaded weights.pth directly -- nothing else in this cell will run")
elif os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt["best_val_loss"]
    print("resuming from checkpoint: epoch %d, best val loss so far %.4f" % (start_epoch, best_val_loss))
else:
    print("no checkpoint found -- starting fresh from epoch 0")

# Two submission-format checkpoints (plain state_dict, matching what
# load_model() expects), plus the richer checkpoint.pt above for resume.
#  - weights.pth is overwritten after EVERY epoch, no matter what.
#  - weights_best.pth is only overwritten when validation loss actually
#    improves, guarding against overfitting late in a long run.
# Early stopping (PATIENCE) is what actually ends the run -- once val loss
# hasn't improved for PATIENCE epochs, weights_best.pth already holds the
# best model and there's nothing more to gain by continuing.
epochs_since_improvement = 0
if not SKIP_TRAINING:
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss = 0.0
        opt.zero_grad()
        for step, (y, cbcr) in enumerate(train_loader):
            y, cbcr = y.to(device), cbcr.to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                pred = model(y)
                pred_rgb = ycbcr_to_rgb_torch(y, pred[:, 0:1], pred[:, 1:2])
                target_rgb = ycbcr_to_rgb_torch(y, cbcr[:, 0:1], cbcr[:, 1:2])
                l1 = color_weighted_l1(pred, cbcr, target_rgb)
                perceptual = vgg_loss_fn(pred_rgb, target_rgb)
                loss = l1 + PERCEPTUAL_WEIGHT * perceptual
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
            train_loss += loss.item() * y.size(0)
        train_loss /= len(train_ds)

        model.eval()
        val_loss = 0.0
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            for y, cbcr in val_loader:
                y, cbcr = y.to(device), cbcr.to(device)
                pred = model(y)
                pred_rgb = ycbcr_to_rgb_torch(y, pred[:, 0:1], pred[:, 1:2])
                target_rgb = ycbcr_to_rgb_torch(y, cbcr[:, 0:1], cbcr[:, 1:2])
                l1 = color_weighted_l1(pred, cbcr, target_rgb)
                perceptual = vgg_loss_fn(pred_rgb, target_rgb)
                val_loss += (l1 + PERCEPTUAL_WEIGHT * perceptual).item() * y.size(0)
        val_loss /= len(val_ds)
        scheduler.step(val_loss)

        print("epoch %2d | lr %.2e | train loss %.4f | val loss %.4f" % (
            epoch, opt.param_groups[0]["lr"], train_loss, val_loss))

        torch.save(model.state_dict(), WEIGHTS_PATH)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_since_improvement = 0
            torch.save(model.state_dict(), WEIGHTS_BEST_PATH)
            print("  (new best -- also saved to weights_best.pth)")
        else:
            epochs_since_improvement += 1

        torch.save({
            "model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "best_val_loss": best_val_loss,
        }, CHECKPOINT_PATH)

        if PATIENCE > 0 and epochs_since_improvement >= PATIENCE:
            print("early stopping: no val loss improvement for %d epochs (best %.4f, at epoch %d) -- "
                  "weights_best.pth already holds the best model, nothing more to gain by continuing."
                  % (PATIENCE, best_val_loss, epoch - PATIENCE))
            break

    # Submit the best epoch, not just the last one -- also copy the final
    # weights.pth next to this notebook (the FIXED submission path
    # load_model() expects by default), not just on Drive.
    shutil.copyfile(WEIGHTS_BEST_PATH, WEIGHTS_PATH)
    shutil.copyfile(WEIGHTS_PATH, "weights.pth")
    print("best val loss: %.4f -- weights.pth is ready for the cells below" % best_val_loss)